# Phase 6b — LoRA + Metadata + Text Fusion (Kaggle)
Combines the two untested pieces: LoRA-adapted image encoder (Phase 6) + metadata/text fusion (Phase 3/5).

**IMPORTANT CAVEAT (carried over from Phase 5a):** including disease-description TEXT risks the same label-leakage problem found earlier -- each class has one fixed text description, so a trainable classifier can shortcut through it instead of learning genuine image/metadata signal. This notebook trains three variants side by side so the leakage effect is visible and separable, not hidden:
1. **LoRA image + metadata** (no text -- the safe, non-leaking comparison)
2. **LoRA image + metadata + text** (the requested full combination -- interpret cautiously)
3. Reference numbers from earlier phases for context

Report variant 1 as the credible "does LoRA+metadata help" result. Report variant 2 only with the leakage caveat explicitly stated, same as Phase 5a.

In [ ]:
%pip install -q setuptools packaging ftfy regex tqdm peft scikit-learn
%pip install -q -U torchao
%pip install -q git+https://github.com/openai/CLIP.git
!git clone --depth 1 https://github.com/Michi-3000/EyeCLIP.git || true

import pathlib
clip_path = pathlib.Path("EyeCLIP/eyeclip/clip.py")
clip_text = clip_path.read_text()
clip_text = clip_text.replace("from pkg_resources import packaging", "import packaging")
clip_path.write_text(clip_text)

import sys
sys.path.insert(0, "EyeCLIP")

import torch, eyeclip
import torch.nn as nn
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from peft import LoraConfig, get_peft_model

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)  # must say cuda -- if cpu, turn on GPU accelerator in notebook settings

CKPT_PATH = "/kaggle/input/eyeclip-checkpoint/eyeclip_visual.pt"          # CHANGE to your actual path
IMAGE_ROOT = Path("/kaggle/input/datasets/faruq05/rop-vl/ROP-VL/Image")   # confirmed working path
SPLIT_DIR = Path("/kaggle/input/datasets/faruq05/rop-vl/ROP-VL/split")    # confirmed working path
DISEASE_DESC_PATH = "/kaggle/input/datasets/faruq05/rop-vl/disease_description.xlsx"  # CHANGE if path differs

META_NUMERIC = ["gestational_age", "birth_weight"]
META_CATEGORICAL = ["gender", "plurality", "delivery_mode"]

In [ ]:
model, preprocess = eyeclip.load("ViT-B/32", device=DEVICE, jit=False)
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
state = ckpt.get("model_state_dict") or ckpt.get("state_dict") or ckpt
model.load_state_dict(state, strict=False)
model.eval()
print("EyeCLIP loaded")

TARGET_MODULES = ["out_proj", "c_fc", "c_proj"]  # confirmed from Phase 6's module inspection
lora_config = LoraConfig(r=8, lora_alpha=16, target_modules=TARGET_MODULES, lora_dropout=0.05, bias="none")
lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()
trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
assert trainable > 0, "LoRA attached to 0 trainable parameters -- check TARGET_MODULES"

## Load splits, encode text descriptions, build metadata matrices
Category-to-folder mapping confirmed from Phase 6 (CSV has lowercase 'normal', folder is 'Normal').

In [ ]:
train_df = pd.read_csv(SPLIT_DIR / "train.csv")
val_df   = pd.read_csv(SPLIT_DIR / "val.csv")
test_df  = pd.read_csv(SPLIT_DIR / "test.csv")

classes_sorted = sorted(train_df["categories"].unique())
class_to_idx = {c: i for i, c in enumerate(classes_sorted)}
NUM_CLASSES = len(classes_sorted)

CATEGORY_TO_FOLDER = {
    "normal": "Normal", "Stage 1 ROP": "Stage 1 ROP", "Stage 2 ROP": "Stage 2 ROP",
    "Stage 3 ROP": "Stage 3 ROP", "Stage 4 ROP": "Stage 4 ROP", "Stage 5 ROP": "Stage 5 ROP",
    "A-ROP": "A-ROP", "Laser-treated ROP": "Laser-treated ROP",
}
missing_map = set(classes_sorted) - set(CATEGORY_TO_FOLDER.keys())
assert not missing_map, f"classes with no folder mapping: {missing_map}"

# text: one description per class, encoded once with EyeCLIP's frozen text encoder
desc_df = pd.read_excel(DISEASE_DESC_PATH)
assert hasattr(eyeclip, "tokenize"), "eyeclip module has no tokenize() -- check text API"
tokens = eyeclip.tokenize(desc_df["description"].tolist(), truncate=True).to(DEVICE)
with torch.no_grad():
    text_embeddings = model.encode_text(tokens).float().cpu().numpy()
class_to_text_emb = dict(zip(desc_df["disease"], text_embeddings))
missing_text = set(classes_sorted) - set(class_to_text_emb.keys())
assert not missing_text, f"classes with no text description: {missing_text}"

# metadata: fit scaler/encoder on train only
num_scaler = StandardScaler().fit(train_df[META_NUMERIC].fillna(train_df[META_NUMERIC].median()))
cat_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(train_df[META_CATEGORICAL].astype(str))

def build_metadata_matrix(df):
    num_part = num_scaler.transform(df[META_NUMERIC].fillna(train_df[META_NUMERIC].median()))
    cat_part = cat_encoder.transform(df[META_CATEGORICAL].astype(str))
    return np.concatenate([num_part, cat_part], axis=1).astype(np.float32)

M_train, M_val, M_test = build_metadata_matrix(train_df), build_metadata_matrix(val_df), build_metadata_matrix(test_df)
META_DIM = M_train.shape[1]
print("metadata dim:", META_DIM)

In [ ]:
class ROPMultimodalDataset(Dataset):
    """Row -> (preprocessed image tensor, metadata vector, text vector, label idx).
    Metadata/text arrays are pre-aligned to df's row order (built outside, indexed by position).
    """
    def __init__(self, df, image_root, preprocess, class_to_idx, meta_matrix, class_to_text_emb):
        self.df = df.reset_index(drop=True)
        self.image_root = image_root
        self.preprocess = preprocess
        self.class_to_idx = class_to_idx
        self.meta_matrix = meta_matrix
        self.class_to_text_emb = class_to_text_emb

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        folder = CATEGORY_TO_FOLDER[row["categories"]]
        img = Image.open(self.image_root / folder / row["image"]).convert("RGB")
        x_img = self.preprocess(img)
        x_meta = self.meta_matrix[i]
        x_text = self.class_to_text_emb[row["categories"]]
        y = self.class_to_idx[row["categories"]]
        return x_img, x_meta, x_text, y

train_ds = ROPMultimodalDataset(train_df, IMAGE_ROOT, preprocess, class_to_idx, M_train, class_to_text_emb)
val_ds   = ROPMultimodalDataset(val_df,   IMAGE_ROOT, preprocess, class_to_idx, M_val,   class_to_text_emb)
test_ds  = ROPMultimodalDataset(test_df,  IMAGE_ROOT, preprocess, class_to_idx, M_test,  class_to_text_emb)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2)

## Model: LoRA image encoder (trainable adapters only) + metadata MLP + optional text, fused
`use_text` toggles between the two variants without duplicating the training loop.

In [ ]:
IMG_DIM = 512

class MetadataMLP(nn.Module):
    def __init__(self, meta_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(meta_dim, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, out_dim))
    def forward(self, m):
        return self.net(m)

class MultimodalHead(nn.Module):
    """Fuses LoRA image embedding + metadata embedding (+ optional text embedding) -> class logits."""
    def __init__(self, img_dim, meta_dim, text_dim, num_classes, use_text):
        super().__init__()
        self.use_text = use_text
        self.meta_mlp = MetadataMLP(meta_dim, img_dim)
        fused_dim = img_dim * 2 + (text_dim if use_text else 0)
        self.fusion = nn.Sequential(nn.Linear(fused_dim, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, num_classes))
    def forward(self, img_emb, meta_raw, text_raw=None):
        meta_emb = self.meta_mlp(meta_raw)
        parts = [img_emb, meta_emb]
        if self.use_text:
            parts.append(text_raw)
        return self.fusion(torch.cat(parts, dim=1))

## Shared training/eval function (runs once per variant: with and without text)

In [ ]:
def train_and_evaluate(use_text, n_epochs=10, lr=1e-4):
    # fresh LoRA adapter + head each run so variants don't leak trained weights into each other
    local_lora_model = get_peft_model(model, lora_config)
    head = MultimodalHead(IMG_DIM, META_DIM, IMG_DIM, NUM_CLASSES, use_text).to(DEVICE)

    class_counts = train_df["categories"].value_counts().reindex(classes_sorted).values
    class_weights = torch.tensor(1.0 / np.maximum(class_counts, 1), dtype=torch.float32, device=DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    params = list(p for p in local_lora_model.parameters() if p.requires_grad) + list(head.parameters())
    optimizer = torch.optim.Adam(params, lr=lr)

    best_val_f1, best_head_state, best_lora_state = -1.0, None, None

    for epoch in range(n_epochs):
        local_lora_model.train(); head.train()
        total_loss = 0.0
        for x_img, x_meta, x_text, y in train_loader:
            x_img, x_meta, x_text, y = x_img.to(DEVICE), x_meta.to(DEVICE), x_text.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            img_emb = local_lora_model.encode_image(x_img).float()
            logits = head(img_emb, x_meta, x_text if use_text else None)
            loss = criterion(logits, y)
            loss.backward(); optimizer.step()
            total_loss += loss.item() * x_img.size(0)

        local_lora_model.eval(); head.eval()
        val_preds, val_labels = [], []
        with torch.no_grad():
            for x_img, x_meta, x_text, y in val_loader:
                x_img, x_meta, x_text = x_img.to(DEVICE), x_meta.to(DEVICE), x_text.to(DEVICE)
                img_emb = local_lora_model.encode_image(x_img).float()
                preds = head(img_emb, x_meta, x_text if use_text else None).argmax(dim=1).cpu().numpy()
                val_preds.extend(preds); val_labels.extend(y.numpy())
        _, _, val_f1, _ = precision_recall_fscore_support(val_labels, val_preds, average=None, zero_division=0)
        val_macro_f1 = val_f1.mean()
        print(f"[use_text={use_text}] epoch {epoch}  loss={total_loss/len(train_ds):.4f}  val_macro_f1={val_macro_f1:.4f}")

        if val_macro_f1 > best_val_f1:
            best_val_f1 = val_macro_f1
            best_head_state = {k: v.clone() for k, v in head.state_dict().items()}
            best_lora_state = {k: v.clone() for k, v in local_lora_model.state_dict().items() if "lora_" in k}

    head.load_state_dict(best_head_state)
    local_lora_model.load_state_dict(best_lora_state, strict=False)

    local_lora_model.eval(); head.eval()
    test_preds, test_labels = [], []
    with torch.no_grad():
        for x_img, x_meta, x_text, y in test_loader:
            x_img, x_meta, x_text = x_img.to(DEVICE), x_meta.to(DEVICE), x_text.to(DEVICE)
            img_emb = local_lora_model.encode_image(x_img).float()
            preds = head(img_emb, x_meta, x_text if use_text else None).argmax(dim=1).cpu().numpy()
            test_preds.extend(preds); test_labels.extend(y.numpy())

    acc = accuracy_score(test_labels, test_preds)
    _, _, f1_per_class, support = precision_recall_fscore_support(
        test_labels, test_preds, labels=range(NUM_CLASSES), average=None, zero_division=0)
    return {"accuracy": acc, "macro_f1": f1_per_class.mean(), "best_val_f1": best_val_f1}, f1_per_class, support

## Run both variants

In [ ]:
print("=== Variant 1: LoRA image + metadata (no text -- credible result) ===")
result_no_text, f1_no_text, support_no_text = train_and_evaluate(use_text=False)
print(result_no_text)

In [ ]:
print("=== Variant 2: LoRA image + metadata + text (leakage risk -- interpret cautiously) ===")
result_with_text, f1_with_text, support_with_text = train_and_evaluate(use_text=True)
print(result_with_text)

In [ ]:
comparison = pd.DataFrame([
    {"representation": "image_only_phase1_frozen", "accuracy": 0.3063, "macro_f1": 0.2469},
    {"representation": "image_plus_metadata_phase3_frozen", "accuracy": 0.3188, "macro_f1": 0.2174},
    {"representation": "lora_image_only_phase6", "accuracy": 0.2375, "macro_f1": 0.165851},
    {"representation": "lora_plus_metadata", "accuracy": result_no_text["accuracy"], "macro_f1": result_no_text["macro_f1"]},
    {"representation": "lora_plus_metadata_plus_text_LEAKAGE_RISK", "accuracy": result_with_text["accuracy"], "macro_f1": result_with_text["macro_f1"]},
])
print(comparison.to_string(index=False))
comparison.to_csv("/kaggle/working/phase6b_lora_metadata_text_comparison.csv", index=False)

print("\nPer-class F1, no-text variant:")
for c, f, s in zip(classes_sorted, f1_no_text, support_no_text):
    print(f"  {c:20s}  F1={f:.3f}  n={s}")

print("\nPer-class F1, with-text variant (check for suspiciously perfect scores -- sign of leakage):")
for c, f, s in zip(classes_sorted, f1_with_text, support_with_text):
    print(f"  {c:20s}  F1={f:.3f}  n={s}")

## Interpretation notes
- Compare `lora_plus_metadata` against `lora_image_only_phase6` (0.2375/0.1659) and `image_plus_metadata_phase3_frozen` (0.3188/0.2174): this tells you whether combining LoRA + metadata recovers what LoRA alone lost, and whether it beats frozen+metadata.
- If `lora_plus_metadata_plus_text` scores near-ceiling (like Phase 5a's 99%), that confirms the same label-leakage mechanism -- report it only with the caveat, never as a genuine trimodal result.
- Do not average variant 1 and variant 2 together in any summary table -- they answer different, non-comparable questions.
- This is an extension beyond the frozen plan's original phases -- log it in the plan's Change Log with today's date and reason ("testing LoRA+metadata combination not specified in original phases, motivated by Phase 6's negative LoRA-alone result").